# Desafio Ciência e Governança de Dados

Desenvolvido por Estevão Augusto da Fonseca Santos, Graduando em Ciência de Computação

6° Período da Universidade Federal de Lavras

## Objetivos

"Como poderíamos avaliar e prever/visualizar os agentes/fenômenos que mais causam impactos socioeconômicos no Brasil?". Essa é a pergunta proposta pelo desafio. Para respondê-la, será preciso adquirir, organizar, explorar e visualizar os dados necessários para criar uma resposta à ele. O notebook "1_coleta_preparacao_dados.ipynb" está focado na coleta e preparação de dados.

In [108]:
import pandas as pd                 # Biblioteca para manipulação e análise de dados
import basedosdados as bd           # Biblioteca para acessar o datalake público do site BasedosDados
import os                           # Biblioteca para interação com arquivos e diretórios a nivel sistema operacional
from dotenv import load_dotenv      # Biblioteca para carregar variáveis de ambiente de arquivos .env
import requests                     # Biblioteca para realizar requisiçoes HTTP
from pathlib import Path            # Biblioteca para a manipulação de caminhos do sistema a nivel orientado a objetos
import gzip                         # Biblioteca para compressão e decompressão de dados usando o formato gzip
import shutil                       # Biblioteca para manipulação de arquivos e diretorios

In [109]:
import sys                          # Biblioteca para acessar variaveis e funções que interagem fortemente com o interpretador
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))  # Adiciona raiz

# Define constantes que armazenam os caminhos que serão utilizados neste notebook
from config_path import RAW_DATA_DIRECTORY_PATH, PROCESSED_DATA_DIRECTORY_PATH, DATA_DIRECTORY_PATH

In [110]:
load_dotenv()  # Carrega os valores no arquivo .env

GOOGLE_CLOUD_ID_PROJECT = os.getenv("GOOGLE_CLOUD_ID_PROJECT") # Coloca o ID do projeto do Google Cloud numa constante

In [111]:
# Caso a pasta 'data' tenha sido excluida, o notebook cria ela
if not os.path.isdir(DATA_DIRECTORY_PATH):
    os.mkdir(DATA_DIRECTORY_PATH)
    os.mkdir(RAW_DATA_DIRECTORY_PATH)
    os.mkdir(PROCESSED_DATA_DIRECTORY_PATH)

## Obtençao de Dados e Preparação de Dados

#### Dados Geográficos do Brasil

##### Dados Geográficos do Brasil de Escala Municipal a até Nacional

In [112]:
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Geograficos_Brasil_Inteiro.ods"):
    url = "https://geoftp.ibge.gov.br/organizacao_do_territorio/estrutura_territorial/areas_territoriais/2024/AR_BR_RG_UF_RGINT_RGI_MUN_2024.ods"
    output = Path(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Geograficos_Brasil_Inteiro.ods")

    r = requests.get(url)
    r.raise_for_status()

    output.write_bytes(r.content)

##### Tamanho Geográfico UF do Brasil

In [113]:
# Aqui realiza-se a leitura de um arquivo de excel sobre dados territoriais do brasil, com o foco no território dos estados
df_uf = pd.read_excel(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Geograficos_Brasil_Inteiro.ods", "AR_BR_UF_2024") 

df_uf.describe() # descreve o que é o df_uf
df_uf.info() # revela as colunas existentes e suas propriedades
df_uf.dropna(how='any', inplace=True)     # remove linhas totalmente vazias
df_uf.dropna(axis=1, how='all', inplace=True)  # remove colunas totalmente vazias
df_uf['ano'] = 2024
df_uf['NM_UF_SIGLA'] = df_uf['NM_UF_SIGLA'].astype(str)

# Converte o arquivo Excel para CSV para uso futuro
df_uf.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_uf_2024.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CD_UF        28 non-null     object 
 1   NM_UF        27 non-null     object 
 2   NM_UF_SIGLA  27 non-null     object 
 3   AR_UF_2024   27 non-null     float64
dtypes: float64(1), object(3)
memory usage: 1.1+ KB


##### Tamanho Geográfico Municipial do Brasil 

In [114]:
# Aqui realiza-se a leitura de um arquivo de excel sobre dados territoriais do brasil, com o foco no território dos municipios
df_mun = pd.read_excel(f"{RAW_DATA_DIRECTORY_PATH}/Dados_Geograficos_Brasil_Inteiro.ods", "AR_BR_MUN_2024")

df_mun.describe() # descreve o que é o df_mun
df_mun.info() # revela as colunas existentes e suas propriedades

df_mun.dropna(how='any', inplace=True)     # remove linhas totalmente vazias
df_mun.dropna(axis=1, how='all', inplace=True)  # remove colunas totalmente vazias
df_mun['ano'] = 2024
df_mun['NM_UF_SIGLA'] = df_mun['NM_UF_SIGLA'].astype(str)

# Converte o arquivo Excel para CSV para uso futuro
df_mun.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_municipios_2024.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5575 entries, 0 to 5574
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CD_UF        5574 non-null   object 
 1   NM_UF        5573 non-null   object 
 2   NM_UF_SIGLA  5573 non-null   object 
 3   CD_MUN       5573 non-null   float64
 4   NM_MUN       5573 non-null   object 
 5   AR_MUN_2024  5573 non-null   float64
dtypes: float64(2), object(4)
memory usage: 261.5+ KB


##### Identificadores de Municipios do Brasil

In [115]:
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/traducao_municipios.csv"):
    url = "https://basedosdados.org/api/tables/downloadTable?p=YnJfYmRfZGlyZXRvcmlvc19icmFzaWw=&q=bXVuaWNpcGlv&d=dHJ1ZQ==&s=ZnJlZQ=="
    output = Path(f"{RAW_DATA_DIRECTORY_PATH}/br_bd_diretorios_brasil_municipio.csv.gz")

    # Realizamos um pedido do arquivo csv
    r = requests.get(url)
    r.raise_for_status()

    # Ao receber ele, nós escrevemos seu conteudo em bytes
    output.write_bytes(r.content)
    
    # O arquivo está compactado em Gz, logo temos de descompacta-lo
    with gzip.open(f'{RAW_DATA_DIRECTORY_PATH}/br_bd_diretorios_brasil_municipio.csv.gz', 'rb') as entrada:
        with open(f'{RAW_DATA_DIRECTORY_PATH}/traducao_municipios.csv', 'wb') as saida:
            shutil.copyfileobj(entrada, saida)
            
    os.remove(output)

In [116]:
# Realiza-se a leitura de um arquivo CSV que contem dados que identificam Estados e Múnicipios de Brasil
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/traducao_municipios.csv")

# Seleciona-se as colunas desejadas para a integração de dados
df = df[df.columns.intersection(["id_municipio", "id_uf","sigla_uf","nome_uf","nome_regiao"])]

# Escreve o novo dataframe gerado num arquivo separado para consultas futuras
df.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/trad_municipio_tratados.csv", index=False)
df.head()

,id_municipio,id_uf,sigla_uf,nome_uf,nome_regiao
0,5101837,51,MT,Mato Grosso,Centro-Oeste
1,1100809,11,RO,Rondônia,Norte
2,1100338,11,RO,Rondônia,Norte
3,1100205,11,RO,Rondônia,Norte
4,1101104,11,RO,Rondônia,Norte


### População Brasileira

In [117]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV sobre a população a nivel municipal

query = """
  SELECT
    dados.ano as ano,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.sexo as sexo,
    dados.grupo_idade as grupo_idade,
    dados.populacao as populacao
FROM `basedosdados.br_ms_populacao.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2016 AND ano <= 2020
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")
df.head()

,ano,id_municipio,id_municipio_nome,sexo,grupo_idade,populacao
0,2016,2402006,Caicó,feminino,0-4 anos,1922
1,2016,2807501,Tomar do Geru,feminino,0-4 anos,514
2,2016,2913101,Ibititá,feminino,0-4 anos,614
3,2016,3101805,Alpercata,feminino,0-4 anos,240
4,2016,3114550,Carneirinho,feminino,0-4 anos,288


### Produto Interno Bruto (PIB) Por Municipio

In [118]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do PIB por Municipio em 2021

query = """
  SELECT
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.ano as ano,
    dados.pib as pib,
    dados.impostos_liquidos as impostos_liquidos,
    dados.va as va,
    dados.va_agropecuaria as va_agropecuaria,
    dados.va_industria as va_industria,
    dados.va_servicos as va_servicos,
    dados.va_adespss as va_adespss
FROM `basedosdados.br_ibge_pib.municipio` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2016 AND ano <= 2020
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio_original.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  
  df.rename(columns={
    'va': 'valor_adicionado_precos_correntes_total',
    'va_agropecuaria': 'valor_adicionado_correntes_agropecuaria',
    'va_industria': 'valor_adicionado_correntes_industria',
    'va_servicos': 'valor_adicionado_correntes_servicos',
    'va_adespss': 'valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social' 
  }, inplace=True)
  
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio_original.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio_original.csv")
df.head()

,id_municipio,id_municipio_nome,ano,pib,impostos_liquidos,valor_adicionado_precos_correntes_total,valor_adicionado_correntes_agropecuaria,valor_adicionado_correntes_industria,valor_adicionado_correntes_servicos,valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social
0,1100809,Candeias do Jamari,2016,399703000,18421000,381283000,69940000,88361000,84077000,138904000
1,1200252,Epitaciolândia,2016,412261000,57022000,355239000,46281000,14949000,194692000,99317000
2,1200385,Plácido de Castro,2016,242578000,10593000,231985000,69876000,14138000,38687000,109284000
3,1301852,Iranduba,2016,625010000,30880000,594130000,180368000,64155000,158151000,191457000
4,1302306,Jutaí,2016,197969000,3851000,194118000,84457000,7547000,32010000,70104000


### Produto Interno Bruto (PIB) Por UF

In [119]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do PIB por UF em 2020

query = """
  SELECT
    dados.ano as ano,
    dados.id_uf AS id_uf,
    diretorio_id_uf.sigla AS id_uf_sigla,
    diretorio_id_uf.nome AS id_uf_nome,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    dados.pib as pib,
    dados.impostos_liquidos as impostos_liquidos,
    dados.va as va,
    dados.va_agropecuaria as va_agropecuaria,
    dados.va_industria as va_industria,
    dados.va_servicos as va_servicos,
    dados.va_adespss as va_adespss
FROM `basedosdados.br_ibge_pib.uf` AS dados
LEFT JOIN (SELECT DISTINCT id_uf,sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_id_uf
    ON dados.id_uf = diretorio_id_uf.id_uf
LEFT JOIN (SELECT DISTINCT sigla,nome  FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
    ON dados.sigla_uf = diretorio_sigla_uf.sigla
WHERE ano >= 2016 AND ano <= 2020
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf_original.csv"):
    df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
    
    df.rename(columns={
        'va': 'valor_adicionado_precos_correntes_total',
        'va_agropecuaria': 'valor_adicionado_correntes_agropecuaria',
        'va_industria': 'valor_adicionado_correntes_industria',
        'va_servicos': 'valor_adicionado_correntes_servicos',
        'va_adespss': 'valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social' 
    }, inplace=True)
    
    df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf_original.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf_original.csv")
df.head()

,ano,id_uf,id_uf_sigla,id_uf_nome,sigla_uf,sigla_uf_nome,pib,impostos_liquidos,valor_adicionado_precos_correntes_total,valor_adicionado_correntes_agropecuaria,valor_adicionado_correntes_industria,valor_adicionado_correntes_servicos,valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social
0,2016,11,RO,Rondônia,RO,Rondônia,39460358978,4075523460,35384835521,4914567473,6572650973,14004784331,9892832744
1,2017,11,RO,Rondônia,RO,Rondônia,43516147492,4235099382,39281048108,5876784259,8193983928,14450296420,10759983506
2,2018,11,RO,Rondônia,RO,Rondônia,44913978486,4654189015,40259789474,5731718700,7063046287,16082571950,11382452532
3,2019,11,RO,Rondônia,RO,Rondônia,47091335805,5053962411,42037373390,5852814369,6936573756,17482423180,11765562096
4,2020,11,RO,Rondônia,RO,Rondônia,51598741456,5360626779,46238114681,6891411669,8285675423,19060688172,12000339417


### Criação de um dataset com dados de todos os múnicipios e suas informações

Código abaixo consiste na integração dos datasets gerados até agora a fim de um arquivo CSV que contenha dados de todos os múnicipios, nisso inclui questões como identificadores (para integração de dados futura) valores economicos, demográficos e populacionais

In [ ]:
# Lendo o arquivo pip_por_municipio.csv
df_pip_por_municipio = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio_original.csv")

In [121]:
# Lendo o arquivo tamanho_municipios_2024.csv
df_tamanho_municipios = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_municipios_2024.csv")

# Lendo o arquivo populacao_brasileira.csv
df_populacao_brasileira = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")

# Pegando as colunas desejadas do arquivo a fim de utiliza-las para a integração de varios dados
df_tamanho_municipios = df_tamanho_municipios[df_tamanho_municipios.columns.intersection(["CD_MUN", "AR_MUN_2024", "NM_UF_SIGLA", "NM_UF"])]

In [122]:
# Tratando dos dados a fim de gerar um arquivo CSV que contenha informaçoes sobre as populações de um municipio, seu pib, valor adicionado a servicos, etc
pop_municipio = df_populacao_brasileira.groupby(['id_municipio', 'ano'])['populacao'].sum().reset_index()
pop_municipio.rename(columns={'populacao': 'populacao_total'}, inplace=True)
pop_municipio = pop_municipio.merge(df_pip_por_municipio, on=['id_municipio', 'ano'], how='inner')

pop_municipio = pd.merge(pop_municipio, df_tamanho_municipios, left_on=["id_municipio"], right_on="CD_MUN", how="inner")

# pop_municipio.drop("CD_MUN", axis=1, inplace=True)

# pop_municipio.drop(columns={ 'ano_y' }, inplace=True)
# pop_municipio.rename(columns={ 'ano_x': 'ano'}, inplace=True)
# Arquivo CSV abaixo contem diversos dados de múnicipios
pop_municipio.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_mun.csv", index=False)


In [123]:
pop_municipio.head()

,id_municipio,ano,populacao_total,id_municipio_nome,pib,impostos_liquidos,valor_adicionado_precos_correntes_total,valor_adicionado_correntes_agropecuaria,valor_adicionado_correntes_industria,valor_adicionado_correntes_servicos,valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social,NM_UF,NM_UF_SIGLA,CD_MUN,AR_MUN_2024
0,1100015,2016,23614,Alta Floresta D'Oeste,478217000,23202000,455014000,166143000,31598000,114541000,142733000,Rondônia,RO,1100015.0,7067.127
1,1100015,2017,23392,Alta Floresta D'Oeste,485374000,25125000,460249000,169623000,27342000,108357000,154926000,Rondônia,RO,1100015.0,7067.127
2,1100015,2018,23167,Alta Floresta D'Oeste,498980000,28255000,470725000,165892000,26060000,123502000,155271000,Rondônia,RO,1100015.0,7067.127
3,1100015,2019,22945,Alta Floresta D'Oeste,495775000,29351000,466424000,162668000,14468000,130314000,158973000,Rondônia,RO,1100015.0,7067.127
4,1100015,2020,22728,Alta Floresta D'Oeste,570242000,35109000,535133000,203394000,20715000,150233000,160791000,Rondônia,RO,1100015.0,7067.127


In [124]:
# Apagando variaveis auxiliares para economizar memoria
del df_pip_por_municipio

### Criação de um dataset com dados de todos os UFs e suas informações

Código abaixo consiste na integração dos datasets gerados até agora a fim de um arquivo CSV que contenha dados de todos os UFs, nisso inclui questões como identificadores (para integração de dados futura) valores economicos, demográficos e populacionais

In [125]:
# Lendo os arquivos CSV
df_tamanho_uf = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_uf_2024.csv")
df_tamanho_uf = df_tamanho_uf[["NM_UF_SIGLA", "AR_UF_2024"]]
df_pip_por_uf = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf_original.csv")

In [127]:
populacao_por_estado = pop_municipio.groupby('NM_UF_SIGLA', as_index=False)['populacao_total'].sum()
populacao_por_estado.rename(columns={'populacao_total': 'populacao_estado'}, inplace=True)

df_pip_por_uf = pd.merge(df_pip_por_uf, df_tamanho_uf, left_on="id_uf_sigla", right_on="NM_UF_SIGLA", how="left")
df_pip_por_uf = pd.merge(df_pip_por_uf, populacao_por_estado, left_on="id_uf_sigla", right_on="NM_UF_SIGLA", how="inner")

# Arquivo CSV abaixo contem diversos dados de UFs
df_pip_por_uf.drop(columns = { 'NM_UF_SIGLA_x', 'NM_UF_SIGLA_y'}, inplace=True)
df_pip_por_uf.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_uf.csv", index=False)

In [128]:
df_pip_por_uf.head()

,ano,id_uf,id_uf_sigla,id_uf_nome,sigla_uf,sigla_uf_nome,pib,impostos_liquidos,valor_adicionado_precos_correntes_total,valor_adicionado_correntes_agropecuaria,valor_adicionado_correntes_industria,valor_adicionado_correntes_servicos,valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social,AR_UF_2024,populacao_estado
0,2016,11,RO,Rondônia,RO,Rondônia,39460358978,4075523460,35384835521,4914567473,6572650973,14004784331,9892832744,237754.171,8786763
1,2017,11,RO,Rondônia,RO,Rondônia,43516147492,4235099382,39281048108,5876784259,8193983928,14450296420,10759983506,237754.171,8786763
2,2018,11,RO,Rondônia,RO,Rondônia,44913978486,4654189015,40259789474,5731718700,7063046287,16082571950,11382452532,237754.171,8786763
3,2019,11,RO,Rondônia,RO,Rondônia,47091335805,5053962411,42037373390,5852814369,6936573756,17482423180,11765562096,237754.171,8786763
4,2020,11,RO,Rondônia,RO,Rondônia,51598741456,5360626779,46238114681,6891411669,8285675423,19060688172,12000339417,237754.171,8786763


In [129]:
# Apagando variaveis auxiliares para economizar na memoria
del df_pip_por_uf
del df_tamanho_uf

### Indice de Gini

In [ ]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV do Indice de Gini a nível Estadual
# Dataset foi criado em 2022

query = """
  SELECT
    dados.id_uf as id_uf,
    dados.ano as ano,
    dados.gini_pib as gini_pib,
    dados.gini_va_agro as gini_va_agro,
    dados.gini_va_industria as gini_va_industria,
    dados.gini_va_servicos as gini_va_servicos,
    dados.gini_va_adespss as gini_va_adespss
FROM `basedosdados.br_ibge_pib.gini` AS dados
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/indice_de_gini_uf.csv")
df.head()

,id_uf,ano,gini_pib,gini_va_agro,gini_va_industria,gini_va_servicos,gini_va_adespss
0,21,2016,0.716927,0.469070,0.886096,0.808752,0.525547
1,24,2016,0.800925,0.656937,0.883038,0.881091,0.647395
2,26,2016,0.790401,0.615988,0.893473,0.852449,0.600472
3,29,2016,0.776509,0.566998,0.901530,0.841915,0.566643
4,14,2016,0.732934,0.382918,0.848332,0.863358,0.650846


### Censo 2022 - Alfabetização por Sexo, Raça e Grupo de Idade

#### Obtenção do Dataset

In [131]:
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizadas_por_sexo_cor_ou_raca_e_idade_original.ods"):
    url = "https://sidra.ibge.gov.br/geratabela?format=ods&name=pessoas_15_anos_ou_mais_total_e_as_alfabetizadas_por_sexo_cor_ou_ra%C3%A7a_e_grupos_idade_original.ods&terr=NC&rank=-&query=t/9542/n6/all/v/allxp/p/all/c59/all/c2/6794/c86/95251/c287/100362/l/v,p%2Bc59%2Bc2,t%2Bc86%2Bc287"
    output = Path(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizadas_por_sexo_cor_ou_raca_e_idade_original.ods")

    r = requests.get(url)
    r.raise_for_status()

    output.write_bytes(r.content)
teste = pd.read_excel(f"{RAW_DATA_DIRECTORY_PATH}/alfabetizadas_por_sexo_cor_ou_raca_e_idade_original.ods", "Tabela")

#### Tratamento do Dataset

In [132]:
teste.dropna(how='any', inplace=True)     # remove linhas totalmente vazias
teste.drop(columns={"Unnamed: 2", "Unnamed: 3"}, inplace=True)
teste.rename(columns={
        "Tabela 9542 - Pessoas de 15 anos ou mais de idade, total e as alfabetizadas, por sexo, cor ou raça e grupos de idade" : "id_municipio",
        "Unnamed: 1" : "nome_municipio",
        "Unnamed: 4" : "total_alfabetizados_e_nao_alfabetizados",
        "Unnamed: 5" : "total_alfabetizados",
        "Unnamed: 6" : "total_nao_alfabetizados",
    }, inplace=True)

teste['UF'] = teste['nome_municipio'].str.extract(r'([A-Z]{2})')
teste.reset_index(inplace=True)
teste.drop(columns={"index"}, inplace=True)

In [133]:
teste

,id_municipio,nome_municipio,total_alfabetizados_e_nao_alfabetizados,total_alfabetizados,total_nao_alfabetizados,UF
0,1100015,Alta Floresta D'Oeste (RO),16867,15448,1419,RO
1,1100023,Ariquemes (RO),75729,71249,4480,RO
2,1100031,Cabixi (RO),4255,3822,433,RO
3,1100049,Cacoal (RO),69419,65056,4363,RO
4,1100056,Cerejeiras (RO),12512,11530,982,RO
...,...,...,...,...,...,...
5565,5222005,Vianópolis (GO),11911,11131,780,GO
5566,5222054,Vicentinópolis (GO),6941,6307,634,GO
5567,5222203,Vila Boa (GO),3188,2779,409,GO
5568,5222302,Vila Propício (GO),4609,3949,660,GO


In [134]:
teste.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/alfabetizacao_processada.csv", index=False)
# Apagando variavel auxiliar para economizar memoria
del teste

### Sinopses Estatísticas da Educação Básica - Sexo Raça Cor

#### Obtenção do dataset

In [135]:
# Código realiza a busca de um dataset especifico do datalake publico do BaseDosDados
# Dataset selecionado foi um arquivo CSV da Educação Básica, a qual conta com o total de matrículas por município para todas as etapas de ensino, sexo e raça/cor
# Dataset foi criado em 2024

query = """
  SELECT
    dados.ano as ano,
    dados.sigla_uf as sigla_uf,
    dados.id_municipio AS id_municipio,
    diretorio_id_municipio.nome AS id_municipio_nome,
    dados.etapa_ensino as etapa_ensino,
    dados.sexo as sexo,
    dados.raca_cor as raca_cor,
    dados.quantidade_matricula as quantidade_matricula
FROM `basedosdados.br_inep_sinopse_estatistica_educacao_basica.sexo_raca_cor` AS dados
LEFT JOIN (SELECT DISTINCT id_municipio,nome  FROM `basedosdados.br_bd_diretorios_brasil.municipio`) AS diretorio_id_municipio
    ON dados.id_municipio = diretorio_id_municipio.id_municipio
WHERE ano >= 2016 AND ano <= 2020
"""

# Caso o arquivo não exista, ele acessa o datalake e o escreve como arquivo CSV na pasta raw
if not os.path.exists(f"{RAW_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv"):
  df = bd.read_sql(query = query, billing_project_id = GOOGLE_CLOUD_ID_PROJECT)
  df.to_csv(f"{RAW_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv", index=False)

# Visualização das primeiras cinco linhas do dataframe
df = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/educacao_basica_sexo_raca_cor.csv")
df.head()

,ano,sigla_uf,id_municipio,id_municipio_nome,etapa_ensino,sexo,raca_cor,quantidade_matricula
0,2019,AC,1200328,Jordão,Educação Especial – Classes Comuns,Feminino,Amarela,1
1,2019,AC,1200252,Epitaciolândia,Educação Especial – Classes Comuns,Masculino,Amarela,0
2,2019,AC,1200351,Marechal Thaumaturgo,Educação Especial – Classes Comuns,Masculino,Amarela,1
3,2019,AC,1200807,Porto Acre,Educação Especial – Classes Comuns,Feminino,Amarela,0
4,2019,AC,1200807,Porto Acre,Educação Especial – Classes Comuns,Masculino,Amarela,0


#### Tratamento do dataset

In [ ]:
# Esse código trata o dataset de forma a criar um arquivo CSV que contabiliza a quantidade de matriculas
# agrupadas entre grupos de etapa_ensino, sexo e ano

# calcula a quantidade de matriculas por ensino dividio pelo etapa_ensino, sexo e ano
matriculas_por_ensino_sexo = df.groupby(
    ['etapa_ensino', 'sexo', 'ano'], as_index=False
)['quantidade_matricula'].sum()

# Renomear a coluna para algo mais claro
matriculas_por_ensino_sexo.rename(columns={'quantidade_matricula': 'total_matriculas'}, inplace=True)

print(matriculas_por_ensino_sexo)

# Escrever no arquivo csv
matriculas_por_ensino_sexo.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/quantidade_total_matriculas_alfabetizacao.csv", 
                                  index=False)
del matriculas_por_ensino_sexo

                          etapa_ensino       sexo   ano  total_matriculas
0   Educação Especial – Classes Comuns   Feminino  2016            287843
1   Educação Especial – Classes Comuns   Feminino  2017            318309
2   Educação Especial – Classes Comuns   Feminino  2018            353710
3   Educação Especial – Classes Comuns   Feminino  2019            371930
4   Educação Especial – Classes Comuns   Feminino  2020            385507
..                                 ...        ...   ...               ...
85                Ensino Médio Regular  Masculino  2016           3874152
86                Ensino Médio Regular  Masculino  2017           3813226
87                Ensino Médio Regular  Masculino  2018           3719920
88                Ensino Médio Regular  Masculino  2019           3607595
89                Ensino Médio Regular  Masculino  2020           3656094

[90 rows x 4 columns]


In [137]:
# Calcula a quantidade de matriculas baseado em grupos de certos criterios (municipio, etapa de ensino, e sexo)
matriculas_por_ensino_sexo = df.groupby(
    ['id_municipio', 'id_municipio_nome', 'etapa_ensino', 'sexo','ano'], as_index=False
)['quantidade_matricula'].sum()

# Renomear a coluna para algo mais claro
matriculas_por_ensino_sexo.rename(columns={'quantidade_matricula': 'total_matriculas'}, inplace=True)

print(matriculas_por_ensino_sexo)

# Escreve o dataframe num arquivo CSV
matriculas_por_ensino_sexo.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/quantidade_total_matriculas_alfabetizacao_dividida_mun.csv", 
                                  index=False)
del matriculas_por_ensino_sexo

        id_municipio      id_municipio_nome  \
0            1100015  Alta Floresta D'Oeste   
1            1100015  Alta Floresta D'Oeste   
2            1100015  Alta Floresta D'Oeste   
3            1100015  Alta Floresta D'Oeste   
4            1100015  Alta Floresta D'Oeste   
...              ...                    ...   
501295       5300108               Brasília   
501296       5300108               Brasília   
501297       5300108               Brasília   
501298       5300108               Brasília   
501299       5300108               Brasília   

                              etapa_ensino       sexo   ano  total_matriculas  
0       Educação Especial – Classes Comuns   Feminino  2016                46  
1       Educação Especial – Classes Comuns   Feminino  2017                38  
2       Educação Especial – Classes Comuns   Feminino  2018                42  
3       Educação Especial – Classes Comuns   Feminino  2019                48  
4       Educação Especial – Classes

## Geração de Tabelas do Brasil, Minas Gerais e Lavras

### Tabela de Informações Gerais do Brasil

In [138]:
df_populacao = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/populacao_brasileira.csv")
df_pib_uf = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_uf_original.csv")
df_pib_mun = pd.read_csv(f"{RAW_DATA_DIRECTORY_PATH}/pip_por_municipio_original.csv")
df_tamanho_uf = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_uf.csv")
df_tamanho_mun = pd.read_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/tamanho_populacional_mun.csv")

In [139]:
# Muito dos valores da tabela são representados por milhoes, a fim de tornar sua visualização mais facil

# Agregações
populacao_total_em_milhoes = (df_populacao.groupby(['ano'], as_index=False)['populacao'].sum())
tamanho_brasil_km2 = df_tamanho_uf.groupby(['ano'], as_index=False)['AR_UF_2024'].sum()
pib_total_brasil = df_pib_uf.groupby(['ano'], as_index=False)['pib'].sum()
valor_agro = df_pib_uf.groupby(['ano'], as_index=False)['valor_adicionado_correntes_agropecuaria'].sum()
valor_industria = df_pib_uf.groupby(['ano'], as_index=False)['valor_adicionado_correntes_industria'].sum()
valor_servicos = df_pib_uf.groupby(['ano'], as_index=False)['valor_adicionado_correntes_servicos'].sum()
valor_adespss = df_pib_uf.groupby(['ano'], as_index=False)['valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social'].sum()


# --- Cálculo do PIB per capita ---
pib_per_capta_brasil = pd.merge(left=pib_total_brasil, right=populacao_total_em_milhoes, how="inner", left_on="ano", right_on="ano")
pib_per_capta_brasil['pib_per_capta_brasil'] = pib_per_capta_brasil['pib'] / pib_per_capta_brasil['populacao']

# --- Cálculo da densidade populacional ---
densidade_populacional = pd.merge(left=populacao_total_em_milhoes, right=tamanho_brasil_km2, how="inner", left_on="ano", right_on="ano")
densidade_populacional['densidade_populacional'] = densidade_populacional['populacao'] / densidade_populacional['AR_UF_2024']


# --- Merge de todas as informações em um único DataFrame ---
brasil_info = populacao_total_em_milhoes.merge(
    tamanho_brasil_km2, on='ano'
).merge(
    densidade_populacional[['ano', 'densidade_populacional']], on='ano'
).merge(
    pib_total_brasil, on='ano'
).merge(
    pib_per_capta_brasil[['ano', 'pib_per_capta_brasil']], on='ano'
).merge(
    valor_agro, on='ano'
).merge(
    valor_industria, on='ano'
).merge(
    valor_servicos, on='ano'
).merge(
    valor_adespss, on='ano'
)

# Escreve o dataframe num arquivo CSV
brasil_info.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/brasil_info.csv", index=False)

### Tabela de Informações de Minas Gerais

In [140]:
estado_minas_gerais = df_tamanho_uf.loc[df_tamanho_uf["id_uf_sigla"] == "MG", :]
estado_minas_gerais['pib_per_capta_mg'] = estado_minas_gerais.apply(lambda x: x['pib'] / x['populacao_estado'], axis=1)
estado_minas_gerais['densidade_populacional'] = estado_minas_gerais.apply(lambda x: x['populacao_estado'] / x['AR_UF_2024'], axis=1)
estado_minas_gerais['valor_total'] = estado_minas_gerais.apply(
    lambda x:   x['valor_adicionado_correntes_agropecuaria'] + x['valor_adicionado_correntes_industria'] + 
                x['valor_adicionado_correntes_servicos'] + x['valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social'], axis=1)


# Escreve o dataframe num arquivo CSV
estado_minas_gerais.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/mg_info.csv", index=False)

### Tabela de Informações de Lavras

In [141]:
mun_lavras = df_tamanho_mun.loc[df_tamanho_mun["id_municipio"] == 3138203]
mun_lavras['pib_per_capta_lavras'] = mun_lavras.apply(lambda x: x['pib'] / x['populacao_total'], axis=1)
mun_lavras['densidade_populacional'] = mun_lavras.apply(lambda x: x['populacao_total'] / x['AR_MUN_2024'], axis=1)
mun_lavras['valor_total'] = mun_lavras.apply(
    lambda x:   x['valor_adicionado_correntes_agropecuaria'] + x['valor_adicionado_correntes_industria'] + 
                x['valor_adicionado_correntes_servicos'] + x['valor_adicionado_correntes_adm_defesa_edu_saude_seguranca_social'], axis=1)


# Escreve o dataframe num arquivo CSV
mun_lavras.to_csv(f"{PROCESSED_DATA_DIRECTORY_PATH}/lavras_info.csv", index=False)